# Problem Set 2: the W Jacobian peak

Use FindingZ's **Standard Model**, **LHC Run 2**, **W -> e nu (both charges)**,
and **CMS-like detector**. Generate 10,000 events with the full pipeline and retain
Delphes ROOT output. This notebook uses detector objects, not the dilepton-only columns.
The new process preset requires the updated course catalog; no application-code change is needed.

These plots are simulation, not UA1 data. For history, read [UA1 (1983), section 10 and figure 9](https://doi.org/10.1016/0370-2693(83)91177-2).
Run in FindingZ's Jupyter environment with numpy, matplotlib, awkward, uproot and vector installed.
The first plot is an explicitly idealized isotropic decay calculation; the later plots require an actual completed W run.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Set this to MASS 24 in the saved generation param_card before comparing simulation.
MW = 80.4  # GeV; reference value for the ideal illustration

def transverse_mass(pt, met, phi, met_phi):
    return np.sqrt(np.maximum(0.0, 2 * pt * met * (1 - np.cos(phi - met_phi))))

# Simple kinematic checks: parallel vectors, antiparallel vectors, azimuth periodicity.
np.testing.assert_allclose(transverse_mass(40., 40., 0., 0.), 0.)
np.testing.assert_allclose(transverse_mass(40., 40., 0., np.pi), 80.)
np.testing.assert_allclose(transverse_mass(40., 40., 2*np.pi, np.pi), 80.)

# Exact bin probabilities avoid evaluating the divergent density at the endpoint.
def ideal_bin_density(edges, endpoint):
    cdf = 1 - np.sqrt(1 - np.clip(edges / endpoint, 0, 1)**2)
    return np.diff(cdf) / np.diff(edges)

bins_pt = np.arange(0, 102, 2.)
bins_mt = np.arange(0, 142, 2.)
fig, axes = plt.subplots(1, 2, figsize=(10, 3.6), constrained_layout=True)
for ax, edges, endpoint, label in zip(axes, [bins_pt, bins_mt], [MW/2, MW],
                                     [r'$p_{T\ell}$ [GeV]', r'$m_T$ [GeV]']):
    density = ideal_bin_density(edges, endpoint)
    np.testing.assert_allclose(np.sum(density*np.diff(edges)), 1.)
    ax.stairs(density, edges, label='Isotropic decay, zero W transverse momentum')
    ax.axvline(endpoint, color='black', ls='--')
    ax.set(xlabel=label, ylabel='Probability density [1/GeV]')
    ax.legend(fontsize=8)
fig.suptitle('Ideal Jacobian peaks: fixed mass, no cuts or detector effects')
plt.show()


## Find and open your simulated sample
The run list is read from `FINDINGZ_RUN_ROOT`. Copy the ID of your completed **w_enu** run below.
Read its saved `cards/param_card.dat` and set `MW` above to the W mass (MASS 24).
Inspect the saved cuts and event count as well. A MadGraph-only run has no detector ROOT file.


In [ ]:
from findingz.delphes import list_runs, open_run
import awkward as ak

runs = list_runs()
display(runs)


In [ ]:
RUN_ID = ""  # Copy your completed W run ID from the table.
if not RUN_ID:
    raise ValueError("Generate a W sample and enter RUN_ID; no simulated data have been substituted.")
events = open_run(RUN_ID)
if events.manifest.get('config', {}).get('process') != 'w_enu':
    raise ValueError("Choose the w_enu sample for this exercise.")
display(events.manifest.get('config', {}))
print('Saved cards:', events.path.parent.parent / 'cards')


## Reconstruct the transverse mass
Select one electron and no muon. Delphes already applies its reconstruction and isolation requirements.
The missing-momentum vector includes the detector's measurement of the whole event.
Do not replace it by the negative electron momentum: that would force zero reconstructed W recoil.


In [ ]:
electrons, muons, missing = events.electrons, events.muons, events.missing_et
has_objects = (ak.num(electrons) == 1) & (ak.num(muons) == 0) & (ak.num(missing) == 1)
e = ak.firsts(electrons[has_objects])
met_object = ak.firsts(missing[has_objects])
pt, eta, phi = (ak.to_numpy(e[field]) for field in ('pt', 'eta', 'phi'))
met, met_phi = (ak.to_numpy(met_object[field]) for field in ('met', 'phi'))
selected = (pt > 25) & (np.abs(eta) < 2.5) & (met > 25)
selected &= np.isfinite(pt + eta + phi + met + met_phi)
pt, phi, met, met_phi = (x[selected] for x in (pt, phi, met, met_phi))
mt = transverse_mass(pt, met, phi, met_phi)
wpt = np.hypot(pt*np.cos(phi) + met*np.cos(met_phi),
               pt*np.sin(phi) + met*np.sin(met_phi))
print(f'{len(mt)} selected events out of {events.event_count}')
if not len(mt):
    raise ValueError('No events pass: inspect the process, generation cuts and detector output.')


In [ ]:
# Unit-area histograms normalize over ALL selected events, including overflow.
# This is a shape comparison for the unweighted LO sample, not a cross-section prediction.
def shape(ax, values, edges, label):
    if not len(values):
        print(f'No events for {label}')
        return
    counts, _ = np.histogram(values, bins=edges)
    ax.stairs(counts / (len(values)*np.diff(edges)), edges, label=label)
    print(label, 'outside displayed range:', int(np.sum((values < edges[0]) | (values > edges[-1]))))

fig, axes = plt.subplots(1, 2, figsize=(10, 4), constrained_layout=True)
low_recoil = wpt < 10
for ax, values, edges, endpoint, label in zip(
    axes, [pt, mt], [bins_pt, bins_mt], [MW/2, MW],
    [r'$p_{Te}$ [GeV]', r'$m_T$ [GeV]']):
    shape(ax, values, edges, 'Selected W simulation')
    shape(ax, values[low_recoil], edges, r'Also $p_{TW}^{reco}<10$ GeV')
    ax.axvline(endpoint, color='black', ls='--', label='Ideal endpoint')
    ax.set(xlabel=label, ylabel='Probability density [1/GeV]')
    ax.legend(fontsize=8)
fig.suptitle('FindingZ: reconstructed W decay, 2 GeV bins')
plt.show()


## Interpret the result
- Which peak is more sensitive to the W transverse momentum?
- Explain how the cuts change the comparison to the ideal plot. A physical W angular distribution is not generally isotropic.
- Distinguish finite W width from detector resolution when discussing events above the nominal mass.
- Why is this signal-only exercise insufficient for a W discovery or a precision mass fit? Discuss backgrounds, calibration and templates.

The kinematic functions and notebook syntax can be checked without a W run. The detector cells must be run on a completed sample; an ideal plot is not evidence that the simulation pipeline has been validated.
